# **Fase 1: PyTorch e EDOs**
**Objetivo:** dominar o *framework* que será usado em PINNs e conectar os gradientes automáticos (*autograds*) com EDOs.

**Conceitos-chave:**

### Sumário

**Etapa 1.1 PyTorch básico**

*Etapa 1.2 `torch.autograd.grad` vs `.backward()`*

*Etapa 1.3 Aproximação de funções*

*Etapa 1.4: Proto-PINN: EDO sem chamar de PINN*

*Mini-projeto 1: Reconstrução de $H(z)$ a partir de dados de cronômetros cósmicos*

# Etapa 1.1 — PyTorch básico

A ideia principal é recriar a MLP criada na Fase 0, mas utilizando PyTorch.

## 1.1.1 Fundamentos de tensores

In [1]:
import torch
import numpy as np

Criando um tensor a partir de uma lista:

In [2]:
dados = [[1], [2], [3], [4], [5]]
x_dados = torch.tensor(dados)

Criando um tensor a partir de um *array* NumPy:

In [3]:
np_dados = np.array(dados)
x_np_dados = torch.from_numpy(np_dados)

In [4]:
print(f'np_dados =\n{np_dados}\n\n x_np_dados =\n {x_np_dados}')

np_dados =
[[1]
 [2]
 [3]
 [4]
 [5]]

 x_np_dados =
 tensor([[1],
        [2],
        [3],
        [4],
        [5]])


Outras funções de criação de tensores úteis:

In [5]:
x_ones = torch.ones_like(x_dados) # Cria um tensor com mesma estrutura que x_dados e preenchido com 1
x_rand = torch.rand_like(x_dados, dtype=torch.float) # Cria um tensor com mesma estrutura que x_dados e preenchida com valores aleatórios

print(f'x_ones =\n{x_ones}\n\n x_rand =\n {x_rand}')

x_ones =
tensor([[1],
        [1],
        [1],
        [1],
        [1]])

 x_rand =
 tensor([[0.4815],
        [0.0042],
        [0.9778],
        [0.0071],
        [0.6891]])


Podemos criar tensores a partir de uma *forma* específica.

In [6]:
forma = (3, 5) # Tupla que define as dimensões do tensor

tensor_ones = torch.ones(forma)
tensor_zeros = torch.zeros(forma)
tensor_rand = torch.rand(forma)

print(f'tensor_ones =\n{tensor_ones}\n\n tensor_zeros =\n {tensor_zeros}\n\n tensor_rand =\n {tensor_rand}')

tensor_ones =
tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

 tensor_zeros =
 tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

 tensor_rand =
 tensor([[0.6951, 0.4005, 0.7837, 0.0699, 0.3690],
        [0.0173, 0.5229, 0.2522, 0.8478, 0.6201],
        [0.7791, 0.3837, 0.6403, 0.2441, 0.4063]])


Atributos de um tensor:

In [7]:
print(f'A forma de tensor_rand é {tensor_rand.shape}.')
print(f'O tipo de dados de tensor_rand é {tensor_rand.dtype}.')
print(f'O objeto tensor_rand está armazenado em {tensor_rand.device}')

A forma de tensor_rand é torch.Size([3, 5]).
O tipo de dados de tensor_rand é torch.float32.
O objeto tensor_rand está armazenado em cpu


Algumas operações básicas envolvendo tensores:

In [8]:
# Fatiamento padrão do NumPy
print(f'Primeira linha: {tensor_rand[0]}.\n')
print(f'Primeira coluna: {tensor_rand[:, 0]}.\n')
print(f'Última coluna: {tensor_rand[:, -1]}.\n')
tensor_rand[:, 1] = 0
print(tensor_rand)

Primeira linha: tensor([0.6951, 0.4005, 0.7837, 0.0699, 0.3690]).

Primeira coluna: tensor([0.6951, 0.0173, 0.7791]).

Última coluna: tensor([0.3690, 0.6201, 0.4063]).

tensor([[0.6951, 0.0000, 0.7837, 0.0699, 0.3690],
        [0.0173, 0.0000, 0.2522, 0.8478, 0.6201],
        [0.7791, 0.0000, 0.6403, 0.2441, 0.4063]])


In [9]:
# Concatenar tensores em alguma dimensão com torch.cat()

tensor1 = torch.cat([tensor_rand, tensor_ones], dim=1) # Concatena as coluans
tensor2 = torch.cat([tensor_rand, tensor_ones], dim=0) # Concatena as linhas

print(tensor1)
print(tensor2)

tensor([[0.6951, 0.0000, 0.7837, 0.0699, 0.3690, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [0.0173, 0.0000, 0.2522, 0.8478, 0.6201, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [0.7791, 0.0000, 0.6403, 0.2441, 0.4063, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000]])
tensor([[0.6951, 0.0000, 0.7837, 0.0699, 0.3690],
        [0.0173, 0.0000, 0.2522, 0.8478, 0.6201],
        [0.7791, 0.0000, 0.6403, 0.2441, 0.4063],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]])


In [10]:
# Tomar a matriz transposta
tensor1_t = tensor1.T

# Multiplicação matricial

t1 = tensor1 @ tensor1_t # Símbolo @ representa a multiplicação matricial
t2 = torch.matmul(tensor1, tensor1_t) # Função que executa uma multiplicação matricial
t3 = torch.zeros_like(t1)
torch.matmul(tensor1, tensor1_t, out=t3) # Função que executa uma multiplicação matricial e salva no tensor t3

# Os tensores t1, t2 e t3 devem ser iguais
print(t1)
print(t2)
print(t3)

tensor([[6.2386, 5.4978, 6.2104],
        [5.4978, 6.1672, 5.6338],
        [6.2104, 5.6338, 6.2416]])
tensor([[6.2386, 5.4978, 6.2104],
        [5.4978, 6.1672, 5.6338],
        [6.2104, 5.6338, 6.2416]])
tensor([[6.2386, 5.4978, 6.2104],
        [5.4978, 6.1672, 5.6338],
        [6.2104, 5.6338, 6.2416]])


In [11]:
# Multiplicação elemento a elemento

z1 = tensor1 * tensor1 # Via símbolo de operação
z2 = torch.mul(tensor1, tensor1) # Via função
z3 = torch.zeros_like(z1)
torch.mul(tensor1, tensor1, out=z3) # Via função e salvando em z3

# Os tensores z1, z2 e z3 devem ser iguais
print(z1)
print(z2)
print(z3)

tensor([[4.8323e-01, 0.0000e+00, 6.1425e-01, 4.8849e-03, 1.3618e-01, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00],
        [2.9816e-04, 0.0000e+00, 6.3612e-02, 7.1882e-01, 3.8448e-01, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00],
        [6.0693e-01, 0.0000e+00, 4.0999e-01, 5.9572e-02, 1.6507e-01, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00]])
tensor([[4.8323e-01, 0.0000e+00, 6.1425e-01, 4.8849e-03, 1.3618e-01, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00],
        [2.9816e-04, 0.0000e+00, 6.3612e-02, 7.1882e-01, 3.8448e-01, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00],
        [6.0693e-01, 0.0000e+00, 4.0999e-01, 5.9572e-02, 1.6507e-01, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00]])
tensor([[4.8323e-01, 0.0000e+00, 6.1425e-01, 4.8849e-03, 1.3618e-01, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00],
        [2.9816e-04, 0.00

In [12]:
# Se o tensor tiver um único elemento, podemos extrai-lo e usá-lo como um valor numérico em Python usual

agg = tensor1.sum() # Soma todos os elementos do tensor
agg_item = agg.item()
print(agg_item, type(agg_item))

20.724918365478516 <class 'float'>


Operções *in-place* são aquelas em que o resultado é armazenado no próprio operando. São denotadas usando um `_` ao final das funções.

In [13]:
print(tensor1)

tensor1.add_(5)
print(f'\n{tensor1}')

tensor([[0.6951, 0.0000, 0.7837, 0.0699, 0.3690, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [0.0173, 0.0000, 0.2522, 0.8478, 0.6201, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [0.7791, 0.0000, 0.6403, 0.2441, 0.4063, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000]])

tensor([[5.6951, 5.0000, 5.7837, 5.0699, 5.3690, 6.0000, 6.0000, 6.0000, 6.0000,
         6.0000],
        [5.0173, 5.0000, 5.2522, 5.8478, 5.6201, 6.0000, 6.0000, 6.0000, 6.0000,
         6.0000],
        [5.7791, 5.0000, 5.6403, 5.2441, 5.4063, 6.0000, 6.0000, 6.0000, 6.0000,
         6.0000]])


Tensor do PyTorch e *arrays* do NumPy podem compartilhar o mesmo local de armazenamento na memória, fazendo com que mudanças em um ocorram no outro.

In [14]:
t = torch.ones(5)
n = t.numpy()
print(f't: {t}')
print(f'n: {n}')

t.add_(1)
print(f't: {t}')
print(f'n: {n}')

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]
t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


Isso também ocorre se você cria um tensor a partir de uma *array*

In [15]:
np_dados = np.array([[1.0,2.0],[3.0,4.0]])
x_np_dados = torch.from_numpy(np_dados)

print(f'np_dados =\n{np_dados}\n\n x_np_dados =\n {x_np_dados}')

np.add(np_dados, 1.5, out=np_dados)

print(f'\nnp_dados =\n{np_dados}\n\n x_np_dados =\n {x_np_dados}')

np_dados =
[[1. 2.]
 [3. 4.]]

 x_np_dados =
 tensor([[1., 2.],
        [3., 4.]], dtype=torch.float64)

np_dados =
[[2.5 3.5]
 [4.5 5.5]]

 x_np_dados =
 tensor([[2.5000, 3.5000],
        [4.5000, 5.5000]], dtype=torch.float64)


Se quisermos calcular o gradiente da função de perda em relação a um tensor, devemos declarar isso na inicialização de tal tensor usndao o parâmetro `requires_grad`.

In [16]:
dados = [[1], [2], [3], [4], [5]]
x_dados = torch.tensor(dados) # Dados de entrada, não calculamos o gradiente
pesos = torch.rand((1, 5), requires_grad=True) # Tensor dos pesos, calculamos o gradiente
bias = torch.rand(5, requires_grad=True) # Tensor dos bias, calculamos o gradiente

## 1.1.2 Criando uma MLP manual via PyTorch

Usando apenas tensores e operações manuais, vamos recriar a rede neural de duas camadas da Fase 0.

In [17]:
dados = [[1], [2], [3], [4], [5]]

x_dados = torch.tensor(dados, dtype=torch.float) # Dados de entrada
Y = 5 * x_dados - 1

w1 = torch.randn((1, 4), dtype=torch.float, requires_grad=True) # Tensor dos pesos da primeira camada
b1 = torch.randn(4, requires_grad=True) # Tensor dos bias da primeira camada

w2 = torch.randn((4, 1), dtype=torch.float, requires_grad=True) # Tensor dos pesos da segunda camada
b2 = torch.randn(1, dtype=torch.float, requires_grad=True) # Tensor dos bias da segunda camada

In [18]:
eta = 0.001 # Taxa de aprendizado
parada = 10e-4
erro = 1
p = 250
i = 0

while erro > parada:
    H1 = x_dados @ w1 + b1
    z1 = torch.nn.functional.relu(H1)
    y = z1 @ w2 + b2

    loss = torch.nn.functional.mse_loss(y, Y)

    loss.backward()

    # Diz ao PyTorch para ignorar, momentaneamente, que precisamos dos gradientes desses valores
    with torch.no_grad():
        w1 -= eta * w1.grad
        b1 -= eta * b1.grad
        w2 -= eta * w2.grad
        b2 -= eta * b2.grad

    # Os .grad acumulam, precisamos "limpá-los" antes de reiniciar o loop
    w1.grad.zero_()
    b1.grad.zero_()
    w2.grad.zero_()
    b2.grad.zero_()

    if (i+1)%100 == 0 or i == 0:
        print(f'Iteração {i+1}\n MSE = {loss}\n' + 30*'-' + '\n')

    if i == 0:
        old_loss = loss
    elif (i+1)%250 == 0:
        erro = old_loss - loss
        old_loss = loss

    i += 1

print(f'Iteração {i+1}\n MSE = {loss}\n' + 30*'-' + '\n')

Iteração 1
 MSE = 570.504638671875
------------------------------

Iteração 100
 MSE = 176.15603637695312
------------------------------

Iteração 200
 MSE = 134.53103637695312
------------------------------

Iteração 300
 MSE = 106.64015197753906
------------------------------

Iteração 400
 MSE = 87.95182800292969
------------------------------

Iteração 500
 MSE = 75.4296875
------------------------------

Iteração 600
 MSE = 67.03922271728516
------------------------------

Iteração 700
 MSE = 61.417144775390625
------------------------------

Iteração 800
 MSE = 57.650054931640625
------------------------------

Iteração 900
 MSE = 55.12593460083008
------------------------------

Iteração 1000
 MSE = 53.43465042114258
------------------------------

Iteração 1100
 MSE = 52.3013916015625
------------------------------

Iteração 1200
 MSE = 51.54206466674805
------------------------------

Iteração 1300
 MSE = 51.03326416015625
------------------------------

Iteração 1400
 MSE = 5